In [2]:
## Practical exercise
# The code below trains and evaluates a decision tree classifier.
# The analysis contains mistakes and choices that could be improved. Answer to the following questions and correct the code accordingly:

# 1) Is the estimate of the accuracy in the original code biased ? Briefly explain why. You can write the answer to this and the following questions as comments in the code.
# 2) Implement a strategy to reduce the risk of overfitting.
# 3) Is the importance of the features accurately estimated ? Briefly explain why.
# 4) Implement a possible strategy to improve the estimate of the features' accuracy

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# from sklearn.datasets import load_breast_cancer
# from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
# from sklearn.tree import DecisionTreeClassifier, plot_tree
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

# data = load_breast_cancer()
# X = pd.DataFrame(data.data, columns=data.feature_names)
# y = data.target

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.3,
# )

# grid = {
#     "max_depth": [None],
#     "min_samples_leaf": [10,],
#     "ccp_alpha": [0.0],
# }

# search = GridSearchCV(
#     DecisionTreeClassifier(),
#     param_grid=grid,
#     scoring="balanced_accuracy",
#     cv=5,
#     n_jobs=-1,
# )

# search.fit(X_train, y_train)

# model = search.best_estimator_

# y_pred = model.predict(X)

# print("Accuracy:", accuracy_score(y, y_pred))

# importances = pd.Series(model.feature_importances_, index=X.columns)
# importances.sort_values(ascending=False).head(10)

#-----SOLUTION-----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y #also this for reproducible class balanced split
)

grid = { #question2
    "max_depth": [2, 3, 4, 5, None ], #list of different candidates for depth not just one (NOne is unlimited depth)
    "min_samples_leaf": [10],
    "ccp_alpha": [0.0, 0.001, 0.005, 0.01, 0.02],
}
dt_cv =  DecisionTreeClassifier(random_state= 42)
search = GridSearchCV(
    estimator = dt_cv,
    param_grid=grid,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

search.fit(X_train, y_train) #no leakage touches only X train

best_model = search.best_estimator_

y_pred_train_pruned = best_model.predict(X_train) #QUESTION 1
y_pred_test_pruned = best_model.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, y_pred_train_pruned))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test_pruned))

rf = RandomForestClassifier(#Q3 AND Q4
    n_estimators=100, max_depth = 3, random_state= 42)
rf.fit(X_train, y_train)

rf_importances = pd.Series(rf.feature_importances_, index=X.columns)
#not necessary: print("\nRandom Forest importances (top 10):")
print(rf_importances.sort_values(ascending=False).head(10))


#Q1: it is not evaluated the full set X, but divided in Xtrain and Xtest and evaluate SEPARATELY, producing an unbiased generalization estimate
#while the original code is trongly biased because it includes training rows the model alreaady memorized
#Q2: grid search can choose max depth, cost-complexity pruning alpha and minsamplessplit over real grid:
#this will reduce the risk of overfitting by searching for optimal parameters jointly instead of fixing arbitrarily
#Q3: feature importance not accurately estimated, because single decision tree has high variance and is unstable->don't get true relevance of feature
#Q4: use a random forest classifier that decorrelatees trees and spread credits more sensibily (averaging over decorellated trees--> more stable


Accuracy: 0.957286432160804
Accuracy: 0.9181286549707602
worst concave points    0.162482
worst area              0.157122
worst perimeter         0.089025
mean radius             0.080438
worst radius            0.080177
mean concave points     0.077084
mean perimeter          0.071743
mean concavity          0.060789
mean area               0.043547
worst concavity         0.040163
dtype: float64
